# Run analysis

Plots and tables of one or more runs. Each run is a directory written by

```bash
.venv/bin/python scripts/summarize_run.py logs-edge logs-cloud --duration 3600 --out results/<run>
```

with, optionally, the files Locust writes in the same run copied next to them
(`request_timestamps.csv`, `workloadProfile.csv`). List the runs below and run all the cells;
with more than one run the plots compare them (up to three runs).

Times are in seconds from the first job of the run. Only what ends within `DURATION` counts.

In [ ]:
RUNS = {
    "old agents": "../results/run-1",
}
DURATION = 3600        # seconds of the run
BIN = 60               # seconds per point of the time series

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Stages in pipeline order, and their response-time thresholds, from the controller's config.
with open(os.path.join("..", "deploy", "k8s", "base", "controller-config.yaml")) as f:
    THRESHOLDS = {s: c.get("res_time_threshold") for s, c in yaml.safe_load(f)["stages"].items()}
STAGES = list(THRESHOLDS)

# Fixed colours: sites and runs keep the same colour in every plot.
SITE_COLORS = {"edge": "#2a78d6", "cloud": "#eb6834"}
RUN_COLORS = dict(zip(RUNS, ["#2a78d6", "#eb6834", "#1baf7a"]))
OBS_COLORS = dict(zip(["workload", "utilization", "pressure", "queue_length_dominant", "n_instances"],
                      ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
assert len(RUNS) <= 3, "compare at most three runs at a time"
PER_BIN = "minute" if BIN == 60 else f"{BIN:g} s"

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (10, 3.2), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e4e4e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "lines.linewidth": 1.5, "legend.frameon": False, "font.size": 9,
})


def load(path):
    run = {name: pd.read_csv(os.path.join(path, f"{name}.csv"))
           for name in ("jobs", "stages", "replicas", "agents")}
    requests = os.path.join(path, "request_timestamps.csv")
    if os.path.exists(requests):
        r = pd.read_csv(requests)
        r["t"] = r["Timestamp"] - r["Timestamp"].min()      # from Locust's first request
        run["requests"] = r
    profile = os.path.join(path, "workloadProfile.csv")
    if os.path.exists(profile):
        run["profile"] = pd.read_csv(profile, sep=";")
    # Response time of a stage: from the previous stage publishing the job to the end.
    s = run["stages"]
    s["response_time"] = s["wait"].fillna(0) + s["service_time"]
    return run


runs = {name: load(path) for name, path in RUNS.items()}
for name, run in runs.items():
    print(f"{name}: {len(run['jobs'])} jobs, sites {sorted(run['stages']['site'].dropna().unique())}, "
          f"Locust files: {[k for k in ('requests', 'profile') if k in run] or 'none'}")

## 1. Load

Jobs per `BIN` that reached the gateway (from the pipeline logs: jobs that never finished
are missing) and, if Locust's files are there, the rate it planned to send.

In [ ]:
fig, axes = plt.subplots(len(runs), 1, sharex=True, figsize=(10, 2.6 * len(runs)), squeeze=False)
for ax, (name, run) in zip(axes[:, 0], runs.items()):
    edges = np.arange(0, DURATION + BIN, BIN)
    counts, _ = np.histogram(run["jobs"]["arrival"], bins=edges)
    ax.bar(edges[:-1] / 60, counts, width=BIN / 60 * 0.9, align="edge", color=RUN_COLORS[name],
           label=f"arrived (jobs per {PER_BIN})")
    if "profile" in run:
        p = run["profile"]
        planned = p.groupby(p["Time"] // BIN)["RPS"].sum()
        ax.step(planned.index * BIN / 60, planned.values, where="post", color="#0b0b0b",
                label=f"planned by Locust (jobs per {PER_BIN})")
    ax.set_title(name, loc="left")
    ax.set_ylabel(f"jobs per {PER_BIN}")
    ax.legend(loc="upper right")
axes[-1, 0].set_xlabel("minutes from the start")
plt.tight_layout()

## 2. Job response time

Duration of every job, from the gateway receiving it to the end of the last stage, against
its arrival time; the line is the 95th percentile per `BIN`.

In [ ]:
fig, axes = plt.subplots(len(runs), 1, sharex=True, sharey=True, figsize=(10, 2.8 * len(runs)), squeeze=False)
for ax, (name, run) in zip(axes[:, 0], runs.items()):
    j = run["jobs"][run["jobs"]["outcome"] == "completed"]
    ax.scatter(j["arrival"] / 60, j["duration"], s=8, color=RUN_COLORS[name], alpha=0.6, linewidths=0)
    p95 = j.groupby(j["arrival"] // BIN)["duration"].quantile(0.95)
    ax.plot(p95.index * BIN / 60, p95.values, color="#0b0b0b", label=f"p95 per {PER_BIN}")
    ax.set_title(name, loc="left")
    ax.set_ylabel("job duration (s)")
    ax.legend(loc="upper right")
axes[-1, 0].set_xlabel("arrival (minutes from the start)")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 3.2))
for name, run in runs.items():
    d = np.sort(run["jobs"].loc[run["jobs"]["outcome"] == "completed", "duration"])
    ax.plot(d, np.arange(1, len(d) + 1) / len(d), color=RUN_COLORS[name], label=name)
ax.set_xlabel("job duration (s)")
ax.set_ylabel("fraction of jobs")
ax.set_title("Distribution of the job duration", loc="left")
if len(runs) > 1:
    ax.legend()
plt.tight_layout()

rows = {}
for name, run in runs.items():
    j = run["jobs"]
    done = j[j["outcome"] == "completed"]
    rows[name] = {
        "completed in the run": int(done["ended_in_run"].sum()),
        "completed after the end": int((done["ended_in_run"] == 0).sum()),
        "failed": int((j["outcome"] == "failed").sum()),
        "mean (s)": done["duration"].mean(),
        "p50 (s)": done["duration"].median(),
        "p95 (s)": done["duration"].quantile(0.95),
        "max (s)": done["duration"].max(),
    }
pd.DataFrame(rows).T.round(2)

## 3. Service time of every stage (profiling)

Time each stage spent on a job (end - start in the worker), per site. The means are the
demands to give the training environment.

In [ ]:
table = pd.concat({name: run["stages"] for name, run in runs.items()}, names=["run"]).reset_index(level=0)
summary = (table.groupby(["stage", "site", "run"])["service_time"]
           .describe(percentiles=[0.05, 0.5, 0.95])
           .rename(columns={"count": "n"}))
summary = summary.reindex(STAGES, level="stage")
summary.round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for ax, stage in zip(axes.flat, STAGES):
    data, labels, colors = [], [], []
    for name, run in runs.items():
        s = run["stages"][run["stages"]["stage"] == stage]
        for site in ("edge", "cloud"):
            values = s.loc[s["site"] == site, "service_time"]
            if len(values):
                data.append(values)
                labels.append(site if len(runs) == 1 else f"{site}\n{name}")
                colors.append(SITE_COLORS[site])
    if data:
        box = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=True,
                         flierprops={"markersize": 3, "markeredgecolor": "#8a8984"},
                         medianprops={"color": "#0b0b0b"})
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_edgecolor(color)
        ax.set_xticks(range(1, len(labels) + 1), labels)
    ax.set_title(stage, loc="left")
for ax in axes[:, 0]:
    ax.set_ylabel("service time (s)")
plt.tight_layout()

## 4. Waits per stage

Mean wait of the jobs entering each stage, per `BIN`: queueing, plus the hop from the
previous stage (and the seed copy when it comes from the other site). A wait that grows
during the peaks means the stage did not have enough replicas.

In [ ]:
fig, axes = plt.subplots(2, 3, sharex=True, figsize=(12, 5.2))
for ax, stage in zip(axes.flat, STAGES):
    for name, run in runs.items():
        s = run["stages"][run["stages"]["stage"] == stage]
        w = s.groupby(s["start"] // BIN)["wait"].mean()
        ax.plot(w.index * BIN / 60, w.values, color=RUN_COLORS[name], label=name)
    ax.set_title(stage, loc="left")
for ax in axes[:, 0]:
    ax.set_ylabel("mean wait (s)")
for ax in axes[-1]:
    ax.set_xlabel("minutes from the start")
if len(runs) > 1:
    axes[0, 0].legend()
plt.tight_layout()

## 5. Replicas

Replicas alive on each site (including those shutting down but still working), with the
load on top. One plot per run.

In [ ]:
for name, run in runs.items():
    fig, axes = plt.subplots(len(STAGES) + 1, 1, sharex=True, figsize=(10, 1.5 * (len(STAGES) + 1)))
    edges = np.arange(0, DURATION + BIN, BIN)
    counts, _ = np.histogram(run["jobs"]["arrival"], bins=edges)
    axes[0].bar(edges[:-1] / 60, counts, width=BIN / 60 * 0.9, align="edge", color="#8a8984")
    axes[0].set_ylabel(f"jobs per {PER_BIN}")
    axes[0].set_title(f"{name}: load and replicas alive", loc="left")
    r = run["replicas"]
    for ax, stage in zip(axes[1:], STAGES):
        for site, color in SITE_COLORS.items():
            s = r[(r["stage"] == stage) & (r["site"] == site)]
            if s["alive"].any():
                ax.step(s["t"] / 60, s["alive"], where="post", color=color, label=site)
        ax.set_ylabel(stage, rotation=0, ha="right", va="center")
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    handles = [plt.Line2D([], [], color=c, label=s) for s, c in SITE_COLORS.items()]
    axes[1].legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    axes[-1].set_xlabel("minutes from the start")
    axes[-1].set_xlim(0, DURATION / 60)
    plt.tight_layout()

## 6. Agents

For every stage scaled by an agent: the normalised state it received (all between 0 and 1)
and its answers (the points) with the replicas applied after each one, kept between the
`min` and `max` of the stage. One figure per run and stage.

In [ ]:
for name, run in runs.items():
    a = run["agents"]
    decided = a[a["action"].notna()]
    if decided.empty:
        print(f"{name}: no agent decisions")
        continue
    for (stage, site), d in decided.groupby(["stage", "site"]):
        fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(10, 3.6), height_ratios=[3, 2])
        for key, color in OBS_COLORS.items():
            top.plot(d["t"] / 60, d[f"obs_{key}"], color=color, marker="o", markersize=3, label=key)
        top.set_ylim(-0.05, 1.05)
        top.set_ylabel("state (0-1)")
        top.set_title(f"{name}: agent of {stage} on {site}", loc="left")
        top.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
        bottom.step(d["t"] / 60, d["applied"], where="post", color=SITE_COLORS[site], label="replicas applied")
        bottom.plot(d["t"] / 60, d["action"], "o", color="#0b0b0b", markersize=4, label="agent's answer")
        bottom.set_ylabel("replicas")
        bottom.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        bottom.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
        bottom.set_xlabel("minutes from the start")
        plt.tight_layout()

## 7. Resources and response-time violations

Replica-hours used by every stage on every site within the run (from the replicas alive,
sampled every 5 s), and the share of jobs whose response time at a stage exceeded its
`res_time_threshold` (by more than 10 %, as the original controller counts a violation).

In [ ]:
def replica_hours(run):
    r = run["replicas"].sort_values("t")
    r = r[(r["t"] >= 0) & (r["t"] <= DURATION)].copy()
    r["dt"] = r.groupby(["site", "stage"])["t"].diff().shift(-1).fillna(0)
    return (r["alive"] * r["dt"]).groupby([r["stage"], r["site"]]).sum() / 3600


def violations(run):
    s = run["stages"]
    limit = s["stage"].map(THRESHOLDS) * 1.1
    return (s["response_time"] > limit).groupby(s["stage"]).mean() * 100


hours = pd.DataFrame({name: replica_hours(run) for name, run in runs.items()}).reindex(STAGES, level=0)
viol = pd.DataFrame({name: violations(run) for name, run in runs.items()}).reindex(STAGES)
print("Replica-hours")
display(hours.round(2))
print("Jobs over the threshold (%)")
display(viol.round(1))

## 8. Comparison between runs

The key numbers of every run side by side.

In [ ]:
compare = {}
for name, run in runs.items():
    j = run["jobs"]
    done = j[(j["outcome"] == "completed") & (j["ended_in_run"] == 1)]
    compare[name] = {
        "jobs completed in the run": len(done),
        "failed": int((j["outcome"] == "failed").sum()),
        "job duration p50 (s)": done["duration"].median(),
        "job duration p95 (s)": done["duration"].quantile(0.95),
        "replica-hours": replica_hours(run).sum(),
        **{f"violations {s} (%)": violations(run).get(s, np.nan) for s in ("ffmpeg2", "deepspeech")},
    }
pd.DataFrame(compare).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
per_stage = hours.groupby(level="stage").sum().reindex(STAGES)
width = 0.8 / len(runs)
for i, name in enumerate(runs):
    ax.bar(np.arange(len(STAGES)) + i * width, per_stage[name], width=width * 0.92,
           color=RUN_COLORS[name], label=name)
ax.set_xticks(np.arange(len(STAGES)) + width * (len(runs) - 1) / 2, STAGES)
ax.set_ylabel("replica-hours")
ax.set_title("Resources used per stage (both sites)", loc="left")
if len(runs) > 1:
    ax.legend()
plt.tight_layout()